In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class MLP:
    """
    Classe para implementação de uma Rede Neural Multicamada (MLP)
    Arquitetura: 2-3-3-1 (2 entradas, 3 neurônios na primeira camada,
    3 neurônios na segunda camada, 1 saída)
    """

    def __init__(self, learning_rate=0.1):
        """
        Inicializa a MLP com os parâmetros especificados

        Parâmetros:
            - learning_rate: taxa de aprendizado (padrão = 0.1)
        """
        self.learning_rate = learning_rate

        # Inicialização dos pesos e bias para cada camada
        # Camada 1: entrada (2) -> oculta (3)
        self.W1 = np.random.normal(0, 0.5, (2, 3))
        self.b1 = np.random.normal(0, 0.5, (1, 3))

        # Camada 2: oculta (3) -> oculta (3)
        self.W2 = np.random.normal(0, 0.5, (3, 3))
        self.b2 = np.random.normal(0, 0.5, (1, 3))

        # Camada 3: oculta (3) -> saída (1)
        self.W3 = np.random.normal(0, 0.5, (3, 1))
        self.b3 = np.random.normal(0, 0.5, (1, 1))

        # Armazenar histórico de erros para análise
        self.errors = []
    
    def sigmoid(self, x):
        """
        Função de ativação Sigmoid

        Parâmetros:
            - x: valor ou vetor para aplicar a função sigmoid

        Retorna:
            - Valor transformado pela função sigmoid
        """
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        """
        Derivada da função de ativação Sigmoid

        Parâmetros:
            - x: valor ou vetor (resultado da sigmoid)

        Retorna:
            - Derivada da função sigmoid aplicada a x
        """
        return x * (1 - x)

    def forward(self, X):
        """
        Passo forward da rede neural

        Parâmetros:
            - X: dados de entrada (batch_size x 2)

        Retorna:
            - y_pred: previsão da rede para os dados de entrada
        """
        # Camada 1: entrada -> primeira camada oculta (3 neurônios)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        # Camada 2: primeira camada oculta -> segunda camada oculta (3 neurônios)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)

        # Camada 3: segunda camada oculta -> saída (1 neurônio)
        self.z3 = np.dot(self.a2, self.W3) + self.b3
        self.y_pred = self.sigmoid(self.z3)

        return self.y_pred
    
    def backward(self, X, y):
        """
        Passo backward (backpropagation) para calcular os gradientes e atualizar os pesos

        Parâmetros:
            - X: dados de entrada (batch_size x 2)
            - y: valores alvo (batch_size x 1)
        """
        # Cálculo do erro da saída
        output_error = self.y_pred - y

        # Gradientes para a camada 3 (saída)
        delta_output = output_error * self.sigmoid_derivative(self.y_pred)

        # Atualização dos pesos e bias da camada 3
        dW3 = np.dot(self.a2.T, delta_output)
        db3 = np.sum(delta_output, axis=0, keepdims=True)

        # Cálculo do erro para a camada 2
        error_layer2 = np.dot(delta_output, self.W3.T)
        delta_layer2 = error_layer2 * self.sigmoid_derivative(self.a2)

        # Atualização dos pesos e bias da camada 2
        dW2 = np.dot(self.a1.T, delta_layer2)
        db2 = np.sum(delta_layer2, axis=0, keepdims=True)

        # Cálculo do erro para a camada 1
        error_layer1 = np.dot(delta_layer2, self.W2.T)
        delta_layer1 = error_layer1 * self.sigmoid_derivative(self.a1)

        # Atualização dos pesos e bias da camada 1
        dW1 = np.dot(X.T, delta_layer1)
        db1 = np.sum(delta_layer1, axis=0, keepdims=True)

        # Aplica a atualização dos pesos e biases (descida do gradiente)
        self.W3 -= self.learning_rate * dW3
        self.b3 -= self.learning_rate * db3
    
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
    
    def train(self, X, y, epochs=1000):
        """
        Treina a rede neural por um número específico de épocas

        Parâmetros:
            - X: dados de entrada
            - y: valores alvo
            - epochs: número de épocas para treinamento
        """
        print("Iniciando treinamento da MLP...")
        print(f"Parâmetros: Learning Rate = {self.learning_rate}, Epochs = {epochs}")
        print("-" * 50)

        for epoch in range(epochs):
            # Forward pass
            y_pred = self.forward(X)

            # Cálculo do erro (MSE - Mean Squared Error)
            error = np.mean((y_pred - y) ** 2)

            # Armazena o erro para análise
            self.errors.append(error)
            
            # Backward pass (backpropagation)
            self.backward(X, y)
            
            # Imprimir progresso a cada 100 épocas
            if (epoch + 1) % 100 == 0:
                print(f"Época {epoch + 1}: Erro = {error:.6f}")

        print(f"\nTreinamento concluído!")
        print(f"Erro final: {self.errors[-1]:.6f}")
    
    def predict(self, X):
        """
        Faz previsões com a rede neural treinada

        Parâmetros:
            - X: dados de entrada para previsão

        Retorna:
            - Previsões da rede neural
        """
        return self.forward(X)
